In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("SET TIME ZONE 'UTC'")

bronze_table = "observatorio_dev.bronze.demanda_real"
silver_table = "observatorio_dev.silver.demanda_real"

print("Bronze:", bronze_table)
print("Silver:", silver_table)

In [0]:
last_ingestion_timestamp = (
    spark.table(silver_table)
    .select(F.max("ingestion_timestamp").alias("ultima_ingesta"))
    .first()["ultima_ingesta"]
)

df_bronze = spark.table(bronze_table)

if last_ingestion_timestamp is None:
    print("Silver está vacía. Se procesará todo el histórico de Bronze.")
    df_nuevos = df_bronze
else:
    print("Última ingesta procesada:", last_ingestion_timestamp)

    df_nuevos = df_bronze.filter(
        F.col("ingestion_timestamp") > F.lit(last_ingestion_timestamp)
    )

registros_recibidos = df_nuevos.count()

print(f"Registros Bronze por procesar: {registros_recibidos:,}")

In [0]:
df_transformado = (
    df_nuevos
    .select(
        F.upper(
            F.trim(F.col("codigo_variable"))
        ).alias("codigo_variable"),

        F.to_timestamp("fecha_hora").alias("fecha_hora"),

        F.upper(
            F.trim(F.col("codigo_sic_agente"))
        ).alias("codigo_agente"),

        F.upper(
            F.trim(F.col("tipo_mercado"))
        ).alias("tipo_mercado"),

        F.upper(
            F.trim(F.col("version"))
        ).alias("version"),

        F.regexp_replace(
            F.col("valor").cast("string"),
            ",",
            "."
        ).cast("double").alias("demanda_real_kwh"),

        F.upper(
            F.trim(F.col("unidad_medida"))
        ).alias("unidad_medida"),

        F.upper(
            F.trim(F.col("codigo_duracion"))
        ).alias("codigo_duracion"),

        F.col("source_file_name"),
        F.col("source_file_path"),
        F.col("ingestion_timestamp"),
        F.col("load_date")
    )
    .withColumn(
        "es_demanda_cero",
        F.col("demanda_real_kwh") == F.lit(0)
    )
    .withColumn(
        "silver_created_at",
        F.current_timestamp()
    )
    .withColumn(
        "silver_updated_at",
        F.current_timestamp()
    )
)

In [0]:
condicion_invalida = (
    F.col("codigo_variable").isNull()
    | (F.length("codigo_variable") == 0)
    | F.col("fecha_hora").isNull()
    | F.col("codigo_agente").isNull()
    | (F.length("codigo_agente") == 0)
    | F.col("tipo_mercado").isNull()
    | (F.length("tipo_mercado") == 0)
    | F.col("version").isNull()
    | (F.length("version") == 0)
    | F.col("demanda_real_kwh").isNull()
    | F.col("unidad_medida").isNull()
    | F.col("codigo_duracion").isNull()
)

registros_invalidos = (
    df_transformado
    .filter(condicion_invalida)
    .count()
)

valores_negativos = (
    df_transformado
    .filter(F.col("demanda_real_kwh") < 0)
    .count()
)

valores_cero = (
    df_transformado
    .filter(F.col("es_demanda_cero"))
    .count()
)

print(f"Registros inválidos: {registros_invalidos:,}")
print(f"Valores negativos: {valores_negativos:,}")
print(f"Valores iguales a cero: {valores_cero:,}")

if registros_invalidos > 0:
    raise ValueError(
        f"Se encontraron {registros_invalidos:,} registros inválidos."
    )

if valores_negativos > 0:
    raise ValueError(
        f"Se encontraron {valores_negativos:,} valores negativos."
    )

In [0]:
columnas_clave = [
    "codigo_variable",
    "fecha_hora",
    "codigo_agente",
    "tipo_mercado",
    "codigo_duracion",
    "unidad_medida",
    "version"
]

window_duplicados = (
    Window
    .partitionBy(*columnas_clave)
    .orderBy(
        F.col("ingestion_timestamp").desc_nulls_last(),
        F.col("load_date").desc_nulls_last()
    )
)

df_unico = (
    df_transformado
    .withColumn(
        "numero_fila",
        F.row_number().over(window_duplicados)
    )
    .filter(F.col("numero_fila") == 1)
    .drop("numero_fila")
)

registros_unicos = df_unico.count()

print(f"Registros únicos para MERGE: {registros_unicos:,}")

In [0]:
if registros_unicos == 0:
    print("No existen registros nuevos para procesar.")

else:
    target = DeltaTable.forName(spark, silver_table)

    condicion_merge = """
        target.codigo_variable = source.codigo_variable
        AND target.fecha_hora = source.fecha_hora
        AND target.codigo_agente = source.codigo_agente
        AND target.tipo_mercado = source.tipo_mercado
        AND target.codigo_duracion = source.codigo_duracion
        AND target.unidad_medida = source.unidad_medida
        AND target.version = source.version
    """

    (
        target.alias("target")
        .merge(
            df_unico.alias("source"),
            condicion_merge
        )
        .whenMatchedUpdate(
            condition="""
                NOT (
                    target.demanda_real_kwh
                    <=> source.demanda_real_kwh
                )
            """,
            set={
                "demanda_real_kwh": "source.demanda_real_kwh",
                "es_demanda_cero": "source.es_demanda_cero",
                "source_file_name": "source.source_file_name",
                "source_file_path": "source.source_file_path",
                "ingestion_timestamp": "source.ingestion_timestamp",
                "load_date": "source.load_date",
                "silver_updated_at": "current_timestamp()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "codigo_variable": "source.codigo_variable",
                "fecha_hora": "source.fecha_hora",
                "codigo_agente": "source.codigo_agente",
                "tipo_mercado": "source.tipo_mercado",
                "version": "source.version",
                "demanda_real_kwh": "source.demanda_real_kwh",
                "unidad_medida": "source.unidad_medida",
                "codigo_duracion": "source.codigo_duracion",
                "es_demanda_cero": "source.es_demanda_cero",
                "source_file_name": "source.source_file_name",
                "source_file_path": "source.source_file_path",
                "ingestion_timestamp": "source.ingestion_timestamp",
                "load_date": "source.load_date",
                "silver_created_at": "source.silver_created_at",
                "silver_updated_at": "source.silver_updated_at"
            }
        )
        .execute()
    )

    print("MERGE de demanda real a Silver finalizado.")

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    MIN(fecha_hora) AS fecha_minima,
    MAX(fecha_hora) AS fecha_maxima,
    COUNT(DISTINCT codigo_agente) AS agentes_distintos,
    COUNT(DISTINCT tipo_mercado) AS mercados_distintos,
    COUNT(DISTINCT version) AS versiones_distintas,
    SUM(CASE WHEN es_demanda_cero THEN 1 ELSE 0 END) AS valores_cero
FROM observatorio_dev.silver.demanda_real;

In [0]:
%sql
WITH duplicados AS (
    SELECT
        codigo_variable,
        fecha_hora,
        codigo_agente,
        tipo_mercado,
        codigo_duracion,
        unidad_medida,
        version,
        COUNT(*) AS repeticiones
    FROM observatorio_dev.silver.demanda_real
    GROUP BY
        codigo_variable,
        fecha_hora,
        codigo_agente,
        tipo_mercado,
        codigo_duracion,
        unidad_medida,
        version
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS grupos_duplicados,
    COALESCE(SUM(repeticiones), 0) AS filas_en_grupos,
    COALESCE(SUM(repeticiones - 1), 0) AS filas_excedentes,
    COALESCE(MAX(repeticiones), 0) AS maximo_repeticiones
FROM duplicados;

In [0]:
%sql
SELECT
    tipo_mercado,
    version,
    COUNT(*) AS registros,
    MIN(fecha_hora) AS fecha_minima,
    MAX(fecha_hora) AS fecha_maxima,
    SUM(demanda_real_kwh) AS demanda_kwh
FROM observatorio_dev.silver.demanda_real
GROUP BY
    tipo_mercado,
    version
ORDER BY
    tipo_mercado,
    version;